<a href="https://colab.research.google.com/github/adiltariq234/intelligent-workflow-hub/blob/main/Langgrap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Langgraph**

In [11]:
!pip install -q -U langgraph langchain langchain-groq python-dotenv

In [12]:
from google.colab import userdata

groq_api_key = userdata.get("GROQ_API_KEY")

In [13]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=groq_api_key
)

**Differnt Types of State **

In [14]:
#1) Type Dict
import os
from typing import TypedDict


class State(TypedDict):
  topic : str
  summary : str
  score : int


#2) Pydantic
from pydantic import BaseModel, field_validator

class State(BaseModel):
  topic : str
  score : int
  summary : str =""

  @field_validator('score')
  def score_positive(cls,v):
    if v <0 :
      raise ValueError("Score must be positive")
    return v

#3 Python dataclasses

from dataclasses import dataclass ,field

@dataclass
class State:
  topic : str =""
  score : int = ""
  messages : list =field(default_factory=list)

# 4  langgrap

from langgraph.graph import MessagesState

class State(MessagesState):
  topic : str
  score : int

**Project 1**

In [16]:
import os
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq

# ==========================
# Initialize LLM
# ==========================

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=groq_api_key
)

# ==========================
# Define State
# ==========================

class PipelineState(TypedDict):
    raw_input: str
    edited_text: str
    script_text: str
    final_output: str

# ==========================
# Node 1 - Editor
# ==========================

def editor_node(state: PipelineState) -> dict:
    print("\n===== Executing Editor Node === SST ===")

    prompt = (
        "You are a professional copy editor. "
        "Correct grammar, spelling, punctuation, and improve clarity while preserving "
        "the original meaning and tone. "
        "Return only the polished text.\n\n"
        f"Text:\n{state['raw_input']}"
    )

    response = llm.invoke(prompt)

    return {
        "edited_text": response.content.strip()
    }

# ==========================
# Node 2 - Script Writer
# ==========================

def script_writer_node(state: PipelineState) -> dict:
    print("\n===== Executing Script Writer Node ====")

    prompt = (
        "You are an expert video script writer. "
        "Convert the following text into an engaging YouTube-style video script. "
        "Start with a strong hook, explain naturally, and finish with a memorable ending. "
        "Return only the script.\n\n"
        f"Text:\n{state['edited_text']}"
    )

    response = llm.invoke(prompt)

    return {
        "script_text": response.content.strip()
    }

# ==========================
# Node 3 - Translator
# ==========================

def translator_node(state: PipelineState) -> dict:
    print("\n===== Executing Translator Node ====")

    prompt = (
        "You are a professional translator. "
        "Translate the following video script into natural spoken English. "
        "Keep the meaning, tone, and emotion exactly the same. "
        "Return only the translated script.\n\n"
        f"Script:\n{state['script_text']}"
    )

    response = llm.invoke(prompt)

    return {
        "final_output": response.content.strip()
    }

# ==========================
# Build Graph
# ==========================

graph = StateGraph(PipelineState)

graph.add_node("Editor", editor_node)
graph.add_node("ScriptWriter", script_writer_node)
graph.add_node("Translator", translator_node)

graph.add_edge(START, "Editor")
graph.add_edge("Editor", "ScriptWriter")
graph.add_edge("ScriptWriter", "Translator")
graph.add_edge("Translator", END)

app = graph.compile()

# ==========================
# Run Graph
# ==========================

result = app.invoke(
    {
        "raw_input": """
        ai is changng the wrld.
        it help peple work faster and save time.
        """
    }
)

print("\n========== FINAL OUTPUT ==========\n")
print(result["final_output"])


===== Executing Editor Node === SST ===

===== Executing Script Writer Node ====

===== Executing Translator Node ====

========== FINAL OUTPUT ==========

Imagine being able to get more done in less time, ditching all those tedious tasks, and focusing on what really counts. Sounds too good to be true, right? But the truth is, this is now a reality, all thanks to the power of Artificial Intelligence. AI is completely transforming the way we live and work, and it's changing the world as we know it.

With AI on our side, people can work smarter and faster, saving a ton of time in the process. By taking care of all the repetitive tasks, providing valuable insights, and boosting productivity, AI is freeing us up to focus on the high-value tasks that really matter - the ones that need creativity, empathy, and problem-solving skills.

From virtual assistants to machine learning algorithms, AI is being used in all sorts of industries to simplify processes, improve accuracy, and cut costs. Th